# Pydantic & LLM Tool Validation Guide

This notebook is designed to teach three core concepts:
1. **What Pydantic is** and how to use it for data validation.
2. **How to define and use Tools (Function Calling)** with Large Language Models (LLMs).
3. **How to validate LLM and Tool outputs** back into robust Python structures using Pydantic.

## Part 1: What is Pydantic?

[Pydantic](https://docs.pydantic.dev/) is the most widely used data validation library for Python. It enforces type hints at runtime and provides user-friendly errors when data is invalid.

Let's first install the Google GenAI SDK (which we will use later for the LLM part) and import Pydantic.

In [ ]:
!pip install google-genai pydantic -q

Now, let's define a simple Pydantic model representing a User Profile and see how validation works.

In [ ]:
from pydantic import BaseModel, EmailStr, Field, ValidationError
from typing import List, Optional

# Define the schema using Pydantic
class UserProfile(BaseModel):
    username: str = Field(..., description="Unique username")
    email: str = Field(..., description="User email address")
    age: int = Field(..., ge=0, le=120, description="Age must be between 0 and 120")
    interests: List[str] = Field(default_factory=list, description="List of user hobbies")

# Example 1: Valid Data
try:
    valid_user = UserProfile(
        username="alice123",
        email="alice@example.com",
        age=28,
        interests=["Coding", "Hiking"]
    )
    print("✅ Validation Succeeded!")
    print(valid_user.model_dump_json(indent=2))
except ValidationError as e:
    print("❌ Validation Failed:", e)

✅ Validation Succeeded!
{
  "username": "alice123",
  "email": "alice@example.com",
  "age": 28,
  "interests": [
    "Coding",
    "Hiking"
  ]
}


Let's see what happens when we pass invalid data (e.g. negative age, or missing fields). Pydantic automatically catches these errors before they cause bugs downstream.

In [ ]:
# Example 2: Invalid Data
try:
    invalid_user = UserProfile(
        username="bob456",
        email="bob-invalid-email",  # Invalid format (Pydantic validates types)
        age=-5,                     # Violates ge=0
        interests="Not A List"      # Should be a list
    )
except ValidationError as e:
    print("❌ Validation Caught Expected Errors:")
    print(e)

❌ Validation Caught Expected Errors:
2 validation errors for UserProfile
age
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal
interests
  Input should be a valid list [type=list_type, input_value='Not A List', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/list_type


## Part 2: What are Tools in LLM Calls?

In LLM applications, **Tools** (also known as Function Calling) allow the model to interact with the external world. Instead of trying to guess an answer or execute calculations itself, the LLM can decide to run a python function you provide.

Let's set up a tool that fetches weather information. We will define a Pydantic schema for the output we expect from the tool to guarantee its integrity.

In [ ]:
from google import genai
from google.colab import userdata

# Securely load your API Key
try:
    GOOGLE_API_KEY = userdata.get('gemini_key')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Client initialized successfully!")
except Exception as e:
    print("⚠️ Please configure your GOOGLE_API_KEY in the Colab secrets manager (key icon on left pane).")

Client initialized successfully!


Let's write a mock tool that retrieves weather metrics and structured forecast details. We define a Pydantic model representing the guaranteed response structure of this tool.

In [ ]:
# Pydantic model for our Tool's output
class WeatherReport(BaseModel):
    location: str
    temperature_celsius: float
    condition: str
    humidity_percentage: int

# Define the python function that will act as our tool
def get_current_weather(location: str) -> str:
    """Gets the current weather for a given location."""
    print(f"\n[Tool Executing] Fetching live weather data for: {location}...")

    # In a real app, you would make an API call here.
    # We return a structured JSON string mimicking a database response.
    if "tokyo" in location.lower():
        return '{"location": "Tokyo", "temperature_celsius": 18.5, "condition": "Rainy", "humidity_percentage": 85}'
    else:
        return '{"location": "' + location + '", "temperature_celsius": 22.0, "condition": "Sunny", "humidity_percentage": 45}'

## Part 3: Orchestrating LLM Tool Call & Pydantic Validation

Now we will make a model call to Gemini, provide the tool we just created, and instruct it to answer a query that requires executing our tool. After the LLM uses the tool, we will parse and validate the tool's output using Pydantic.

In [ ]:
import json

# 1. Ask Gemini a question requiring the weather tool
prompt = "What is the weather like in Tokyo right now?"
print(f"User prompt: '{prompt}'")

# We pass our python function directly in the 'config.tools' list
response = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt,
    config={
        'tools': [get_current_weather]
    }
)

# 2. Check if the model wants to call our function
if response.function_calls:
    for call in response.function_calls:
        print(f"\n🤖 LLM decided to call function: '{call.name}' with args: {call.args}")

        # Dynamically execute the function based on the name
        if call.name == 'get_current_weather':
            # Extract arguments sent by the LLM
            arguments = dict(call.args)

            # Execute the local python tool
            raw_tool_output = get_current_weather(**arguments)
            print(f"📦 Raw Tool Output: {raw_tool_output}")

            # 3. Validate the output of our tool using Pydantic!
            try:
                validated_report = WeatherReport.model_validate_json(raw_tool_output)
                print("\n✅ Tool Output Validated successfully using Pydantic!")
                display(validated_report)
            except ValidationError as e:
                print("❌ Tool output failed Pydantic validation:")
                print(e)
else:
    print("LLM did not request any function calls. Response:", response.text)

User prompt: 'What is the weather like in Tokyo right now?'

[Tool Executing] Fetching live weather data for: Tokyo...
LLM did not request any function calls. Response: Right now in Tokyo, it is 18.5°C and rainy, with a humidity of 85%.
